# Chapter 5 — Implementing an MCP Server in Python (Hands-On)

Now we move from theory → to implementation.

You’ll build a minimal MCP server that:

* Exposes tools
* Optionally exposes resources
* Runs locally
* Can be connected to by any MCP client

---

# 1️⃣ What an MCP Server Actually Is

An MCP server is simply:

> A process that speaks the MCP protocol (JSON-RPC) and exposes tools/resources/prompts.

It does NOT contain an LLM.

It only exposes capabilities.

Think:

```text
MCP Server = Tool Microservice
```

---

# 2️⃣ High-Level Structure of MCP Server

Internally it has:

```text
1. Tool registry
2. Resource registry
3. Prompt registry
4. JSON-RPC handler
5. Execution engine
```

---

# 3️⃣ Installing MCP SDK (Python)

Most MCP servers today use official SDKs provided by:

* Anthropic (Model Context Protocol creators)

You typically install:

```bash
pip install mcp
```

(Exact package name may vary depending on SDK version.)

---

# 4️⃣ Minimal MCP Server Example (Tool Only)

Let’s implement a simple server exposing a weather tool.

### Step 1 — Define Tool

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("weather-server")

@mcp.tool()
def get_weather(city: str) -> str:
    """
    Get current weather of a city.
    """
    return f"The weather in {city} is 32°C and sunny."
```

Notice:

* Type hints required
* Docstring required
* Decorator registers tool

This is similar to your agent tool style — but now it lives in server.

---

### Step 2 — Run Server

```python
if __name__ == "__main__":
    mcp.run()
```

Now this server:

* Starts JSON-RPC listener
* Exposes `tools/list`
* Exposes `tools/call`

You don’t implement protocol manually.

SDK handles it.

---

# 5️⃣ What Happens Internally

When client connects:

### Client sends:

```json
{
  "method": "tools/list"
}
```

### Server responds:

```json
{
  "tools": [
    {
      "name": "get_weather",
      "description": "Get current weather of a city.",
      "inputSchema": {
        "type": "object",
        "properties": {
          "city": { "type": "string" }
        }
      }
    }
  ]
}
```

Schema automatically generated from:

* Type hints
* Docstring

Very powerful.

---

# 6️⃣ Adding a Resource

Let’s expose customer data.

```python
@mcp.resource("customer://{customer_id}")
def get_customer(customer_id: str) -> dict:
    return {
        "id": customer_id,
        "name": "John Doe",
        "plan": "Premium"
    }
```

Now agent can read:

```text
customer://123
```

No function call needed.

---

# 7️⃣ Adding a Prompt

```python
@mcp.prompt()
def summarize_customer(customer_info: str) -> str:
    """
    Summarize customer information professionally.
    """
    return f"Please summarize the following:\n{customer_info}"
```

Now prompt is reusable across agents.

---

# 8️⃣ Final Full Example

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("example-server")

@mcp.tool()
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email."""
    return "Email sent successfully"

@mcp.resource("customer://{customer_id}")
def get_customer(customer_id: str) -> dict:
    return {"id": customer_id, "name": "John"}

@mcp.prompt()
def review_code(code: str) -> str:
    """Review code professionally."""
    return f"Review this code:\n{code}"

if __name__ == "__main__":
    mcp.run()
```

That’s a complete MCP server.

---

# 9️⃣ How Server Runs

When started:

```text
python server.py
```

Server:

* Waits for client connection
* Advertises tools/resources/prompts
* Executes requests

It can run:

* Locally
* Over stdio
* Over HTTP
* Inside container

---

# 🔟 Production Architecture Example

```text
MCP Server: filesystem
MCP Server: database
MCP Server: internal APIs
MCP Server: analytics
```

Each maintained by separate team.

Agents connect dynamically.

---

# 1️⃣1️⃣ Security Considerations

MCP server can:

* Restrict tool access
* Validate input
* Sandbox execution
* Log usage
* Authenticate clients

Because tools are now infrastructure.

Not agent logic.

---

# 1️⃣2️⃣ Key Insight

When you build MCP server:

You are building a reusable tool platform.

Any compliant agent can connect.

This decouples:

Tool development from agent development.

Huge architectural win.

---

# Analogy — Building Your Own API Server 🌐

Traditional tools = functions inside app

MCP server = building a REST API service

Instead of:

```python
app.call_function()
```

You now have:

```text
client → API → server → function
```

More scalable.

More reusable.

More modular.

---

# Chapter 5 Summary

You learned:

* How to define MCP server
* How tools are registered
* How resources are exposed
* How prompts are created
* How server runs
* Why this enables scalability

---

